## **CH2 流程控制** (80 min)
### **題型主要包含:**
*   for 迴圈執行次數判斷
*   星星題
*   不可能成立的輸出

---


### **作業上繳設定**

1. **檔案 → 在雲端硬碟中儲存副本**
2. 對老師分享的 `2026檢定培訓營` 按 **「新增雲端硬碟捷徑」** 加到「我的雲端硬碟」（課堂會引導）
3. 確認老師給的是 **編輯者** 權限（否則無法上傳）
4. 填好下方 **DAY**（Day1、Day2…）與 **STUDENT_NAME**（你的姓名）
5. 依序執行 3 個灰色程式格：設定 → 掛載 Drive → **上繳工具**（應顯示 `上繳工具已載入（2026-06-15）`）
6. 每題流程：**寫程式 → 執行作答格 → 執行 `make_submit_button("APCS_3-1")` 格 → 按上繳**

上繳後檔案會出現在：

```
2026檢定培訓營 / Day1 / 王小明 / APCS_3-1.py
```

> `Day1`、姓名等子資料夾**不用事先建立**，第一次上繳會自動產生。
> 若改過答案，要**再執行一次** `make_submit_button(...)` 那格，再按上繳。


In [ ]:
# 老師分享的共用資料夾（捷徑到「我的雲端硬碟」後的路徑，通常不用改）
SUBMIT_BASE = "/content/drive/MyDrive/橘子蘋果/APCS/2026檢定培訓營"

# ===== 請填入 =====
DAY = "Day1"           # 統一格式：Day1、Day2、Day3...
STUDENT_NAME = "王小明"  # 你的姓名


In [ ]:
#@title 首次使用需授權 Google 帳號
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 已掛載")


In [ ]:
#@title 上傳相關設定
import os

import ipywidgets as widgets
from IPython.display import clear_output, display
from IPython import get_ipython

_NOTEBOOK_CACHE = None
SUBMIT_TOOL_VERSION = "2026-06-15"


def _user_ns():
    ip = get_ipython()
    return ip.user_ns if ip is not None else globals()


def _resolve_submit_folder():
    ns = _user_ns()
    base = ns.get("SUBMIT_BASE", "")
    day = ns.get("DAY", "").strip()
    name = ns.get("STUDENT_NAME", "").strip()

    if not base:
        raise ValueError("SUBMIT_BASE 未設定")
    if not os.path.isdir(base):
        raise ValueError(
            f"找不到共用資料夾：{base}\n"
            "請確認已掛載 Drive，且已將「2026檢定培訓營」捷徑加到「我的雲端硬碟」"
        )
    if not day:
        raise ValueError("請填 DAY（例如 Day1）")
    if not day.startswith("Day") or not day[3:].isdigit():
        raise ValueError("DAY 請用 Day1、Day2 格式")
    if not name:
        raise ValueError("請填 STUDENT_NAME（你的姓名）")

    return os.path.join(base, day, name)


def _submit_filename(question_id):
    return f"{question_id}.py"


def _cell_source_text(cell):
    src = cell.get("source", [])
    if isinstance(src, str):
        return src
    return "".join(src)


def _capture_notebook_now():
    """在執行 cell 當下讀取 Colab 畫面上的講義（不能在按鈕 callback 裡讀）。"""
    try:
        from google.colab import _message

        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
        if reply and "ipynb" in reply:
            return reply["ipynb"]
    except Exception:
        pass
    return None


def _extract_answer_code(nb, question_id):
    marker = f"#start-{question_id}"
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = _cell_source_text(cell)
        if marker not in src:
            continue
        lines = []
        for line in src.splitlines():
            stripped = line.strip()
            if stripped == marker or stripped == "#start":
                continue
            lines.append(line)
        code = "\n".join(lines).strip()
        return code + "\n" if code else ""
    raise ValueError(
        f"找不到作答區 {marker}。請確認開啟的是「_上繳」版講義，並在作答格寫好程式。"
    )


def make_submit_button(question_id):
    global _NOTEBOOK_CACHE

    nb = _capture_notebook_now()
    if nb is None:
        print("⚠️ 無法讀取講義。請在 Google Colab 中執行此格。")
        _NOTEBOOK_CACHE = None
    else:
        _NOTEBOOK_CACHE = nb
        try:
            _extract_answer_code(nb, question_id)
        except ValueError as exc:
            print(f"⚠️ {exc}")

    btn = widgets.Button(description="上繳", button_style="success", icon="cloud-upload")
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output(wait=True)
            print("上繳中，請稍候…")
            try:
                if _NOTEBOOK_CACHE is None:
                    print("⚠️ 請先重新執行此格（make_submit_button），再按上繳。")
                    return

                if not os.path.isdir("/content/drive/MyDrive"):
                    print("⚠️ 請先執行上方的「掛載 Google Drive」cell")
                    return

                code = _extract_answer_code(_NOTEBOOK_CACHE, question_id)
                if not code.strip():
                    print("⚠️ 作答區是空的，請先寫程式，再重新執行此格，然後按上繳")
                    return

                target_dir = _resolve_submit_folder()
                os.makedirs(target_dir, exist_ok=True)

                filename = _submit_filename(question_id)
                filepath = os.path.join(target_dir, filename)

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(code)

                print("✅ 上繳成功！")
                print(f"檔案：{filename}")
                print(f"路徑：{filepath}")
            except Exception as exc:
                print(f"❌ 上繳失敗：{exc}")

    btn.on_click(on_submit)
    display(widgets.VBox([btn, output]))


print(f"上繳工具已載入（{SUBMIT_TOOL_VERSION}）")

<br>

### **2-1 for 迴圈執行次數判斷：**
### **例題**
### 在下列程式碼中，請問 for 迴圈的執行次數應為多少？
```python
i = 0
j = 0
for _ in range(1000):
  if i >= 128:
    break
  j = i + 1
  i = i + j + 1
```
*   (A) 3
*   (B) 5
*   (C) 7
*   (D) 9




#### **演算過程**
```python
#1 i=0, j=0:
    j = 0+1 = 1
    i = 0+1+1 = 2
#2 i=2, j=1:
    j = 2+1 = 3
    i = 2+3+1 = 6
#3 i=6, j=3:
    j = 6+1 = 7
    i = 6+7+1 = 14
```

*   是否離開迴圈與 i 值有關
*   每一次的迴圈計算中需要檢查 i 值的變化
*   將執行過程刪減，可以得到「i = 2*i+2」
*   那麼就可以快速計算得到 2 6 14 30 62 126 254 共 7 次




<br>







### **2-2 星星題：**
*   了解遞迴函式執行過程中，變數、計算數值或輸出的變化


#### **例題**
#### 請問底線缺少的程式碼，應該填入何者才會產生下圖排列？
```python
for i in range(8):
  for j in range(i):
    print(str(____), end="")
  print()
```
```python
1
0 1
1 0 1
0 1 0 1
1 0 1 0 1
0 1 0 1 0 1
1 0 1 0 1 0 1
```
*   (A) (i+j-1)%2
*   (B) (i-j+1)%2
*   (C) (2*i+j)%2
*   (D) (i+j)%2

#### **程式碼分析**
*   透過迴圈的規律性產生特定圖案，此種題型稱為星星題（常使用 * 作為文字繪圖的依據）
*   繪圖與迴圈的變數值 i, j 有關係



#### **演算過程**

*   從外層迴圈的變數 i 來看數值變化，會影響到內層迴圈的執行次數

  ```python
  for i in range(8):
      for j in range(i):
  ```

  *   i = 1, 內層迴圈執行 1 次
  *   i = 2, 內層迴圈執行 2 次
  *   i = 3, 內層迴圈執行 3 次
  *   ...
  ```python
  1     # --> i = 1, j = 1
  0 1   # --> i = 2, j = 0, 1
  1 0 1 # --> i = 3, j = 0, 1, 2
  0 1 0 1
  1 0 1 0 1
  0 1 0 1 0 1
  1 0 1 0 1 0 1
  ```
* 觀察輸出的數字，可以發現具有規律性
  *   奇數行的第一個數字都是 1
  *   偶數行的第一個數字都是 0
  * **若要達到 0 1 互換排列的情況，需要使用 %2 的計算呈現**

* 如果 (i+j-1)%2

  ```python
  # (i+j-1)
  0
  1 2
  2 3 4

  # %2
  0
  1 0
  0 1 0
  ```

* 如果 (i-j+1)%2

  ```python
  # (i-j+1)
  2
  3 2
  4 3 2

  # %2
  0
  1 0
  0 1 0
  ```

* 如果 (2*i+j)%2

  ```python
  # (2*i+j)
  2
  4 5
  6 7 8

  # %2
  0
  0 1
  0 1 0
  ```

* **如果 (i+j)%2**

  ```python
  # (i+j)
  1
  2 3
  3 4 5

  # %2
  1
  0 1
  1 0 1
  ```




<br>

### **2-3 不可能成立的輸出：**



#### **例題**
#### 請問 func(n) 不可能會回傳什麼數值？
```python
def func(n):
    if n > 10:
        n = n+5
    while n < 12:
        n = n+1
    if n == 14:
        n = 5
    return n
```

*   (A) 15
*   (B) 16
*   (C) 17
*   (D) 18

#### **程式碼分析**
*   此題為判斷題型，藉由迴圈重複累加或改變變數值的特性。
*   分析和推算不同參數下進入程式邏輯後的計算結果。


#### **演算過程**

#### 觀察 func() 的條件式有三個：
* n > 10
* n < 12
* n == 14

#### 【n > 10 的邏輯推算】
#### 如果 n <= 10，則必定進入 while (n<12) 的迴圈，並且使得任何計算結果皆為 12，也不可能進入 line6 的條件式
```python
def func(n):
    if n > 10:
        n = n + 5
    ...
    ...
    return n
```
* n = 11, return 16
* n = 12, return 17
* n = 13, return 18
* ...

#### 【n <= 10 的邏輯推算】
#### 如果 n <= 10，則必定進入 while (n<12) 的迴圈，並且使得任何計算結果皆為 12，也不可能進入
```python
def func(n):
  ...
  while n < 12:
      n = n+1
  ...
  return n
```
* n = 10, return 12
* n = 9, return 12
* n= 8, return 12
* ...





<br>

<br>